# Alanine dipeptide: molecular dynamics and backbone torsions
Use **ASE + AIMNet2** to follow the same capped alanine dipeptide used in
`ConformerSampling`. Relax one supplied conformer, run Langevin dynamics in vacuum,
and follow its φ and ψ angles. This is an isolated molecule, without solvent.

Run from this directory in the AIMNet2 kernel. Helper source is visible in
`../../demo_tools`; the dynamics and analysis are written below.

**Model choice.** We use [AIMNet2](https://github.com/isayevlab/aimnetcentral) for its molecular-chemistry scope and to match the alanine conformer exercise, allowing us to compare the trajectory with the same model’s PES.

Complete the `...` placeholders in **Your turn** cells. Other cells are provided.


**Your turn.** Create AIMNet2ASE for neutral alanine dipeptide using the aimnet2 alias. Leave the environment and proxy setup unchanged.


In [ ]:
import os, sys, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from ase import Atoms, units
from ase.build import molecule
from ase.io import read, write
from ase.optimize import BFGS
from ase.md.langevin import Langevin
from ase.md.verlet import VelocityVerlet
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary

sys.path.insert(0, str(Path.cwd().parent.parent))
from demo_tools.helpers import start_exercise
from demo_tools.view_mols import view
_, OUTPUT = start_exercise()
torch.set_num_threads(int(os.environ.get('SLURM_CPUS_PER_TASK', '4')))

os.environ['TORCH_COMPILE_DISABLE'] = '1'

os.environ["http_proxy"] = "http://10.73.132.63:8080"
os.environ["https_proxy"] = "http://10.73.132.63:8080"

from aimnet.calculators import AIMNet2ASE
calc = ...  # TODO: create the neutral-molecule calculator

## 1. Reuse and relax a conformer
`CONFORMER` selects a frame from the existing structure file. The zero-based torsion
indices match the conformer exercise and depend on this file's atom ordering.

In [ ]:
CONFORMER = 0
atoms = read('../ConformerSampling/alanine_dipeptide_conformers.xyz', index=CONFORMER)
atoms.info.update(total_charge=0, total_spin=0)
atoms.calc = calc
PHI = [6, 4, 3, 1]
PSI = [3, 4, 6, 8]
assert BFGS(atoms, logfile=str(OUTPUT/'relax.log')).run(fmax=0.02, steps=300)
write(OUTPUT/'relaxed.extxyz', atoms)
view(atoms)

## 2. Run dynamics
The controls below set the temperature, timestep, run length, and save interval.
The simulated duration is `STEPS * DT_FS / 1000` ps.
The thermostat exchanges energy with the molecule, so total energy is not conserved.
Net translation is removed initially; Langevin dynamics uses all Cartesian degrees
of freedom. A short trajectory may remain in its initial conformational basin.
Increase `STEPS` for a longer run; CPU time scales approximately with step count.

**Your turn.** Supply the timestep in ASE units and target temperature to Langevin, then run STEPS steps. The thermostat friction, random seed, and logging are provided.


In [ ]:
DT_FS = 0.5
STEPS = 10000
TEMPERATURE_K = 300
SAVE_EVERY = 10
rng = np.random.default_rng(42)
MaxwellBoltzmannDistribution(atoms, temperature_K=TEMPERATURE_K, rng=rng)
Stationary(atoms)
dyn = Langevin(atoms, ..., temperature_K=...,
               friction=0.01/units.fs, fixcm=False, rng=rng,
               trajectory=str(OUTPUT/'alanine.traj'),
               logfile=str(OUTPUT/'md.log'), loginterval=SAVE_EVERY)
start = time.perf_counter()
...  # TODO: run STEPS integration steps
print(f'Elapsed: {(time.perf_counter()-start)/60:.1f} min')

## 3. Follow φ and ψ
Unwrap angles for continuous time traces; wrap into [−180°, 180°) for the
Ramachandran scatter plot. Points show sampled configurations, not a converged
free-energy surface. This uses the same AIMNet2 calculator as the conformer exercise, so you can
relate the sampled torsions to its potential-energy surface.

In [ ]:
frames = read(OUTPUT/'alanine.traj', index=':')
t = np.arange(len(frames))*SAVE_EVERY*DT_FS/1000
angles = np.array([[a.get_dihedral(*PHI), a.get_dihedral(*PSI)] for a in frames])
wrapped = (angles + 180) % 360 - 180
continuous = np.rad2deg(np.unwrap(np.deg2rad(angles), axis=0))
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(t, continuous[:, 0], label='φ')
ax[0].plot(t, continuous[:, 1], label='ψ')
ax[0].set(xlabel='Time (ps)', ylabel='Unwrapped torsion (degrees)'); ax[0].legend()
points = ax[1].scatter(wrapped[:, 0], wrapped[:, 1], c=t, s=12)
ax[1].set(xlabel='φ (degrees)', ylabel='ψ (degrees)', xlim=(-180, 180), ylim=(-180, 180))
fig.colorbar(points, ax=ax[1], label='Time (ps)')
plt.tight_layout(); plt.show()
np.savetxt(OUTPUT/'torsions.csv', np.column_stack((t, wrapped)), delimiter=',',
           header='time_ps,phi_deg,psi_deg', comments='')
view(frames[::max(1, len(frames)//100)], animate=True)

## 4. Place the MD trajectory on the relaxed PES
Reuse the supplied 2D scan from the conformer exercise. The background is potential
energy relative to the sampled minimum; point color shows MD time. The same zero-based
phi/psi indices and angle convention are used in both notebooks. Points are not joined
across the periodic boundary at +/-180 degrees.

The scan relaxes other coordinates at each fixed torsion pair; MD configurations also
have thermal motion in those coordinates. This is a projection onto the relaxed PES,
not the instantaneous MD energy or a free-energy surface.

In [ ]:
pes_frames = read('../ConformerSampling/reference_results/Relaxed_2D_scan.xyz', index=':')
pes = np.array([[a.info['target_phi'], a.info['target_psi'], a.info['Energy']] for a in pes_frames])
fig, ax = plt.subplots(figsize=(9, 6))
surface = ax.tricontourf(pes[:, 0], pes[:, 1], pes[:, 2] - pes[:, 2].min(),
                         levels=20, cmap='Greys')
fig.colorbar(surface, ax=ax, label='Relative potential energy (eV)', pad=0.02)
points = ax.scatter(wrapped[:, 0], wrapped[:, 1], c=t, cmap='plasma', s=15,
                    edgecolors='black', linewidths=0.2)
fig.colorbar(points, ax=ax, label='MD time (ps)', pad=0.12)
ax.scatter(*wrapped[0], marker='o', s=90, facecolors='none', edgecolors='cyan', label='Start')
ax.scatter(*wrapped[-1], marker='x', s=70, c='cyan', label='End')
ax.set(xlabel='Phi (degrees)', ylabel='Psi (degrees)', xlim=(-180, 180), ylim=(-180, 180),
       title='Alanine dipeptide: MD on the relaxed PES')
ax.legend(); fig.tight_layout()
fig.savefig(OUTPUT/'md_on_pes.png', dpi=160)
plt.show()


## Try and explain
- Does the trajectory explore one basin or several? Compare with the conformer exercise.
- Repeat from another `CONFORMER` or at a higher temperature. Rerun from the preparation cell.
- How would you distinguish local fluctuations from a conformational transition?
- Why is a short gas-phase trajectory insufficient to predict solution conformer populations?